# Addition of IBTrACS attributes to the minimal extended IBTrACS

In [1]:
# Setup environment
import huracanpy
import xarray as xr
import numpy as np

In [2]:
# Script parameters
## Extended IBTrACS file you want to add the attributes to (may be the minimal one or one you already added info to)
IN_FILE = "../files/extended_ibtracs.nc"
## Output file automatically determined. You may change if you like.
OUT_FILE = IN_FILE[:-3]+"_ibattrs"+".nc"
## List of the attributes you want to add from IBTrACs, as they are named in the csv version, in capital letters
ATTRS_TO_ADD = [ 
 'subbasin',
 'name',
 'nature',
 'track_type',
 'dist2land',
 'usa_record',
 'usa_status',
 'usa_sshs',
 'usa_poci',
 'usa_roci',
 'usa_rmw',
 'usa_eye',
 'usa_gust',
]

In [3]:
# Open extended IBTrACS
eib = xr.open_dataset(IN_FILE)

In [4]:
# Open IBTrACS as saved in the creation script
ib = huracanpy.load("../input/ibtracs.csv")

/Users/bourdin/Softs/huracanpy/huracanpy/_data/_csv.py:88: DtypeWarning: Columns (43,52) have mixed types. Specify dtype option on import or set low_memory=False.
  tracks = load_function(filename, **kwargs)


In [5]:
# Transform into dataframes
ib_df = ib.to_dataframe()
teib_df = eib.sel(dataset = "IBTrACS").squeeze().reset_coords().to_dataframe()

In [6]:
# Merge data
M = teib_df[["time","track_id"]].reset_index().merge(ib_df, how = "left", on = ["time", "track_id"])

In [7]:
assert len(M) == len(teib_df) # Check that no duplicate were created

In [8]:
# Transform to xarray and select attributes
M_xr = M.set_index("record").to_xarray()[ATTRS_TO_ADD]

In [9]:
# Merge back into the extended ibtracs
eib_with_attrs = eib.merge(M_xr)

In [10]:
# Visualise new catalogue
eib_with_attrs

<xarray.Dataset> Size: 69MB
Dimensions:     (record: 188964, dataset: 6)
Coordinates:
  * dataset     (dataset) <U17 408B 'IBTrACS' ... 'TRACK-ECMWF-OP-AN'
  * record      (record) int64 2MB 0 1 2 3 4 ... 188960 188961 188962 188963
Data variables: (12/19)
    track_id    (record) <U13 10MB ...
    lon         (dataset, record) float64 9MB ...
    lat         (dataset, record) float64 9MB ...
    name        (record) object 2MB 'UNNAMED' 'UNNAMED' ... 'SARA' 'SARA'
    pres        (dataset, record) float64 9MB ...
    wind10      (dataset, record) float64 9MB ...
    ...          ...
    usa_sshs    (record) float64 2MB nan nan nan nan nan ... -1.0 -3.0 -3.0 -3.0
    usa_poci    (record) float64 2MB nan nan nan ... 1.007e+03 1.007e+03
    usa_roci    (record) float64 2MB nan nan nan nan ... 100.0 100.0 100.0 100.0
    usa_rmw     (record) float64 2MB nan nan nan nan nan ... 50.0 50.0 50.0 50.0
    usa_eye     (record) float64 2MB nan nan nan nan nan ... nan nan nan nan nan
    usa_gust    (record) float64 2MB nan nan nan nan nan ... nan 35.0 nan 35.0

In [11]:
# Save
eib_with_attrs.to_netcdf(OUT_FILE)